## Purpose:
    This script constructs graph-connected route sequences from hexagon-based GPS trajectories and splits them into fixed-length OD route segments for model training
    and validation.

## Input:
    - Hexagon-based GPS trajectory files:
      ./data/sample_processed_inputs/Training/gps_hexa/
      ./data/sample_processed_inputs/Validation/gps_hexa/
    - Hexagon road network graph:
      ./data/new_hexagraph/hexa_network_with_road.gpickle
    - Traveler attribute files:
      ./data/sample_processed_inputs/Training/properties_for_traveler.csv
      ./data/sample_processed_inputs/Validation/properties_for_traveler.csv

## Output:
    - Graph-corrected route files:
      ./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv
      ./data/sample_processed_inputs/Validation/shortcut_route/gps_shortcut.csv

    - Fixed-length route segment files:
      ./data/sample_processed_inputs/Training/shortcut_route/route_split_{k}/
      ./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}/

    - Traveler attribute files with start and end nodes:
      ./data/sample_processed_inputs/Training/shortcut_route/traveler_proper_OD_{k}.csv
      ./data/sample_processed_inputs/Validation/shortcut_route/traveler_proper_OD_{k}.csv

## Main procedures:
    1. Load the hexagon road network graph.
    2. Read hexagon-based GPS trajectory files.
    3. Correct disconnected node pairs using NetworkX shortest path.
    4. Save graph-connected shortcut routes.
    5. Split each route into fixed-length overlapping OD segments for k = 8, 12, and 16.
    6. Attach traveler-level attributes and segment-level start/end nodes to each route segment.

In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import geopandas as gpd
import os
import re
import ast
import torch
import networkx as nx
import pickle
from torch_geometric.utils import from_networkx

## 1. Construct graph-connected routes from GPS-based hexagon sequences

In [2]:
def numerical_sort_key(file):
    return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', file)]

In [3]:
# hexagon_data
with open("./data/new_hexagraph/hexa_network_with_road.gpickle", 'rb') as f:
    G = pickle.load(f)
data = from_networkx(G)

num_nodes = data.num_nodes 
data.x = torch.eye(num_nodes) 
node_feat_dim = data.x.size(1)

node_features = data.x
edge_index = data.edge_index

In [4]:
def fix_path_with_dijkstra(G, path):
    """
    Correct disconnected consecutive node pairs in a route using the shortest path
    on the hexagon road network graph.
    """
    fixed_path = []

    for i in range(len(path) - 1):
        start, end = path[i], path[i+1]

        if G.has_edge(start, end) or start == end:
            fixed_path.append(start)
        else:
            try:
                # Find the shortest path between the nodes and add it to the fixed path
                shortest = nx.shortest_path(G, source=start, target=end)
                fixed_path.extend(shortest[:-1])  # Exclude the last node to avoid duplication in the next iteration
            except nx.NetworkXNoPath:
                print(f"[!] No path between {start} and {end}, skipping...")
                fixed_path.append(start)

    fixed_path.append(path[-1])  # Add Final node
    return fixed_path

In [5]:
folder_Tr = './data/sample_processed_inputs/Training/gps_hexa/'
route_csv_list_Tr = sorted(
    [f for f in os.listdir(folder_Tr) if f.endswith(".csv")],
    key=numerical_sort_key
)

folder_Va = './data/sample_processed_inputs/Validation/gps_hexa/'
route_csv_list_Va = sorted(
    [f for f in os.listdir(folder_Va) if f.endswith(".csv")],
    key=numerical_sort_key
)

In [7]:
fixed_paths = []
name_list = []

for i in range(len(route_csv_list_Tr)):
    gps_pd = pd.read_csv(f'{folder_Tr}{route_csv_list_Tr[i]}')
    gps_route = gps_pd['hexa_code'].tolist()
    fixed = fix_path_with_dijkstra(G, gps_route)
    fixed_paths.append(fixed)
    gps_name = route_csv_list_Tr[i].replace('hexa_tn_gps_coord_', '').replace('.csv', '')
    name_list.append(gps_name)

df = pd.DataFrame({
    'name': name_list,
    'path': fixed_paths
})

df.to_csv(f'./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv', index=None)

fixed_paths = []
name_list = []

for i in range(len(route_csv_list_Va)):
    gps_pd = pd.read_csv(f'{folder_Va}{route_csv_list_Va[i]}')
    gps_route = gps_pd['hexa_code'].tolist()
    fixed = fix_path_with_dijkstra(G, gps_route)
    fixed_paths.append(fixed)
    gps_name = route_csv_list_Va[i].replace('hexa_tn_gps_coord_', '').replace('.csv', '')
    name_list.append(gps_name)

df = pd.DataFrame({
    'name': name_list,
    'path': fixed_paths
})

df.to_csv(f'./data/sample_processed_inputs/Validation/shortcut_route/gps_shortcut.csv', index=None)

## 2. Split graph-connected routes into fixed-length OD segments

In [9]:
df_Tr = pd.read_csv("./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv")
df_Va = pd.read_csv('./data/sample_processed_inputs/Validation/shortcut_route/gps_shortcut.csv')

In [10]:
# Load data
df_Tr = pd.read_csv("./data/sample_processed_inputs/Training/shortcut_route/gps_shortcut.csv")
df_Va = pd.read_csv("./data/sample_processed_inputs/Validation/shortcut_route/gps_shortcut.csv")
df_all = pd.concat([df_Tr, df_Va], ignore_index=True)

parsed_lists = df_all.iloc[:, 1].apply(lambda x: ast.literal_eval(str(x)) if pd.notna(x) else [])
lengths = parsed_lists.apply(len).to_numpy()

# calculate statistics
min_len = np.min(lengths)
max_len = np.max(lengths)
mean_len = np.mean(lengths)
count_over_8 = np.sum(lengths >= 8)

# mean of items with length >= 8
mean_over_8 = np.mean(lengths[lengths >= 8]) if count_over_8 > 0 else 0
print("=== Shortcut route length summary ===")
print(f"Number of routes: {len(lengths)}")
print(f"Minimum length: {min_len}")
print(f"Maximum length: {max_len}")
print(f"Mean length: {mean_len:.2f}")
print(f"Number of routes with length >= 8: {count_over_8}")
print(f"Mean length of routes with length >= 8: {mean_over_8:.2f}")

=== Shortcut route length summary ===
Number of routes: 253
Minimum length: 1
Maximum length: 781
Mean length: 188.53
Number of routes with length >= 8: 226
Mean length of routes with length >= 8: 210.86


In [11]:
for k in range(8, 17, 4):
    window_size = k
    stride = window_size - 1

    # --- Training ---
    for i in range(len(df_Tr)):
        user_name = df_Tr.iloc[i, 0][2:]   # Remove the dataset-specific prefix from the route name to match TRAVELER_ID.
        route = ast.literal_eval(df_Tr.iloc[i, 1])

        j = 0
        start_idx = 0

        while start_idx + window_size <= len(route):  # Discard the remaining sequence if it is shorter than the window size.
            end_idx = start_idx + window_size
            tmp_route = route[start_idx:end_idx]
            df_tmp = pd.DataFrame({
                'path': tmp_route,
                'name': user_name
            })
            out_dir = f"./data/sample_processed_inputs/Training/shortcut_route/route_split_{k}"
            os.makedirs(out_dir, exist_ok=True)
            df_tmp.to_csv(f"{out_dir}/{user_name}_{j}.csv", index=False)

            j += 1
            start_idx += stride

    # --- Validation ---
    for h in range(len(df_Va)):
        user_name = df_Va.iloc[h, 0][2:]
        route = ast.literal_eval(df_Va.iloc[h, 1])

        j = 0
        start_idx = 0

        while start_idx + window_size <= len(route):  # Discard the remaining sequence if it is shorter than the window size.
            end_idx = start_idx + window_size
            tmp_route = route[start_idx:end_idx]
            df_tmp = pd.DataFrame({
                'path': tmp_route,
                'name': user_name
            })
            out_dir = f"./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}"
            os.makedirs(out_dir, exist_ok=True)
            df_tmp.to_csv(f"{out_dir}/{user_name}_{j}.csv", index=False)

            j += 1
            start_idx += stride

## 3. Attach traveler attributes and OD nodes to route segments

In [12]:
properties_Tr = pd.read_csv("./data/sample_processed_inputs/Training/properties_for_traveler.csv")
properties_Va = pd.read_csv("./data/sample_processed_inputs/Validation/properties_for_traveler.csv")

In [13]:
for k in range(8, 17, 4):
    route_split_list_Tr = sorted(
        [f for f in os.listdir(f'./data/sample_processed_inputs/Training/shortcut_route/route_split_{k}') if f.endswith('.csv')],
        key=numerical_sort_key
    )

    route_split_list_Va = sorted(
        [f for f in os.listdir(f'./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}') if f.endswith('.csv')],
        key=numerical_sort_key
    )

    columns = properties_Va.columns
    tmp_df = pd.DataFrame(columns= columns)

    for i in range(len(route_split_list_Va)):
        user_name = route_split_list_Va[i][0:7]   # Extract traveler ID from a route split file name.
        tmp = pd.read_csv(f'./data/sample_processed_inputs/Validation/shortcut_route/route_split_{k}/{route_split_list_Va[i]}')
        start_point = tmp.iloc[0, 0]
        end_point = tmp.iloc[-1, 0]

        # Copy traveler-level attributes.
        tmp_row = properties_Va[properties_Va['TRAVELER_ID'] == user_name].copy()
        # Add segment-level origin and destination nodes.
        tmp_row['start_point'] = start_point
        tmp_row['end_point'] = end_point
        # Append the row to the output dataframe.
        tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)

    tmp_df = tmp_df.drop(columns= "Unnamed: 0")
    tmp_df.to_csv(f'./data/sample_processed_inputs/Validation/shortcut_route/traveler_proper_OD_{k}.csv', index=False)



    columns = properties_Tr.columns
    tmp_df = pd.DataFrame(columns= columns)

    for i in range(len(route_split_list_Tr)):
        user_name = route_split_list_Tr[i][0:7]
        tmp = pd.read_csv(f'./data/sample_processed_inputs/Training/shortcut_route/route_split_{k}/{route_split_list_Tr[i]}')
        start_point = tmp.iloc[0, 0]
        end_point = tmp.iloc[-1, 0]

        # Copy traveler-level attributes.
        tmp_row = properties_Tr[properties_Tr['TRAVELER_ID'] == user_name].copy()
        # Add segment-level origin and destination nodes.
        tmp_row['start_point'] = start_point
        tmp_row['end_point'] = end_point
        # Append the row to the output dataframe.
        tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)

    tmp_df = tmp_df.drop(columns= "Unnamed: 0")
    tmp_df.to_csv(f'./data/sample_processed_inputs/Training/shortcut_route/traveler_proper_OD_{k}.csv', index=False)

    print("finish", k)

/tmp/ipykernel_2240031/3252612817.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)
/tmp/ipykernel_2240031/3252612817.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)


finish 8


/tmp/ipykernel_2240031/3252612817.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)
/tmp/ipykernel_2240031/3252612817.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)


finish 12


/tmp/ipykernel_2240031/3252612817.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)
/tmp/ipykernel_2240031/3252612817.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tmp_df = pd.concat([tmp_df, tmp_row], axis=0, ignore_index=True)


finish 16
